# 06 - QLoRA fine-tune Gemma-2-9B-it as the meta-model

Trains on `artifacts/meta_jsonl/train.jsonl`, evaluates on `val.jsonl`. Seed 42 saves to `artifacts/lora_adapter/` (notebook 08 / `LORA_DIR`). Other seeds save to `artifacts/lora_adapter_seeds/{RUN_SEED}/`.

**One seed per Colab session:** set `RUN_SEED` to `42`, `123`, `2024`, `7`, or `99`. Seed 42 reuses the existing adapter if present. New seeds: `TRAIN_MODE='scratch'`.

**Resume after disconnect:** checkpoints go to `artifacts/lora_train_checkpoints/gemma2_9b_seed_{RUN_SEED}/` on Drive. Re-run from the top; the training cell auto-resumes from the latest `checkpoint-*` folder.

Requirements: a CUDA GPU with ~24 GB VRAM (e.g. A100 40GB or A6000) and an `HF_TOKEN` with Gemma-2 license accepted on HuggingFace.

In [ ]:
%pip install -q -U 'torchao>=0.16.0'
%pip install -q 'transformers>=4.44' 'peft>=0.11' 'trl>=0.9' 'bitsandbytes>=0.43' 'accelerate>=0.33' datasets sentencepiece

In [ ]:
import os, sys, json

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

from tm_research.ensemble.utils_io import (
    META_JSONL_DIR, LORA_DIR, ARTIFACTS_DIR, load_label_map
)
label_map = load_label_map()
print('classes', label_map.class_names)

In [ ]:
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError(
        'Set HF_TOKEN environment variable with a token that has accepted the '
        'Gemma-2 license at https://huggingface.co/google/gemma-2-9b-it. '
        'On Colab: open the key icon in the sidebar, add an HF_TOKEN secret, '
        'grant this notebook access, then re-run setup_colab().'
    )

MODEL_NAME = 'google/gemma-2-9b-it'
MAX_SEQ_LEN = 800
# One seed per Colab session. Known protocol seeds: 42, 123, 2024, 7, 99.
RUN_SEED = 42
# Seed 42 reuses artifacts/lora_adapter/ when present. Set True to train anyway.
FORCE_RETRAIN = False

# Persistent on Drive when repo is on Drive — survives Colab disconnect.
CHECKPOINT_DIR = str(
    paths.persistent_artifacts / 'lora_train_checkpoints' / f'gemma2_9b_seed_{RUN_SEED}'
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SEED_ADAPTER_DIR = ARTIFACTS_DIR / 'lora_adapter_seeds' / str(RUN_SEED)
if RUN_SEED == 42:
    FINAL_ADAPTER_DIR = str(LORA_DIR)
else:
    FINAL_ADAPTER_DIR = str(SEED_ADAPTER_DIR)

SKIP_TRAINING = (
    RUN_SEED == 42
    and not FORCE_RETRAIN
    and (LORA_DIR / 'adapter_config.json').is_file()
)
print('RUN_SEED', RUN_SEED, 'SKIP_TRAINING', SKIP_TRAINING)
print('checkpoint dir (persistent):', CHECKPOINT_DIR)
print('final adapter dir:', FINAL_ADAPTER_DIR)
print('seed adapter dir:', SEED_ADAPTER_DIR)

## Build chat-formatted dataset

Each example becomes a Gemma chat with three turns: `system` (task description), `user` (the structured-token block from step 5), and `assistant` (the `<label>...</label>` completion). We use `tokenizer.apply_chat_template` and let `SFTTrainer` mask everything but the assistant span via the `completion_only` collator.

In [ ]:
from datasets import load_dataset

raw = load_dataset(
    'json',
    data_files={
        'train': str(META_JSONL_DIR / 'train.jsonl'),
        'validation': str(META_JSONL_DIR / 'val.jsonl'),
    },
)
raw

In [ ]:
from tm_research.ensemble.utils_stacking import build_meta_system_prompt, PROMPT_SCHEMA_VERSION, PROMPT_PROB_DECIMALS

# Load system prompt from weights.json if present (written by notebook 05).
# Falls back to building it from the current label_map and weights.
with open(ARTIFACTS_DIR / 'weights.json', 'r', encoding='utf-8') as _wf:
    _weights_payload = json.load(_wf)
weights_for_prompt = _weights_payload['weights']

if _weights_payload.get('system_prompt'):
    SYSTEM_PROMPT = _weights_payload['system_prompt']
    print('Loaded system_prompt from weights.json')
else:
    SYSTEM_PROMPT = build_meta_system_prompt(label_map, weights_for_prompt)
    print('Built system_prompt from build_meta_system_prompt()')

print('\nSystem prompt:\n', SYSTEM_PROMPT)

def to_messages(example):
    messages = [
        {'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + example['prompt']},
        {'role': 'assistant', 'content': example['completion']},
    ]
    return {'messages': messages}

ds = raw.map(to_messages, remove_columns=raw['train'].column_names)
print(ds)
print('first messages:', ds['train'][0]['messages'])

## Load model in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from tm_research.ensemble.bert_oof import _seed_everything

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
if SKIP_TRAINING:
    print('SKIP_TRAINING: not loading Gemma / attaching LoRA.')
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # eager avoids CheckpointError with sdpa + non-reentrant grad checkpointing on Gemma-2 QLoRA.
    ATTN_IMPL = os.environ.get('TM_ATTN_IMPL', 'eager')

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPL,
        token=os.environ['HF_TOKEN'],
    )
    # use_reentrant=True is required for stable backward with 4-bit + LoRA; False often raises
    # "different number of tensors" during recomputation. Set USE_GRADIENT_CHECKPOINTING=False if OOM.
    USE_GRADIENT_CHECKPOINTING = True
    _prepare_kwargs = {'use_gradient_checkpointing': USE_GRADIENT_CHECKPOINTING}
    if USE_GRADIENT_CHECKPOINTING:
        _prepare_kwargs['gradient_checkpointing_kwargs'] = {'use_reentrant': True}
    model = prepare_model_for_kbit_training(model, **_prepare_kwargs)

    _seed_everything(RUN_SEED)
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

## SFTTrainer (assistant-only loss)

In [ ]:
import inspect, shutil
from pathlib import Path
from trl import SFTTrainer, SFTConfig

# ---------------------------------------------------------------------------
# Training mode — set this before running the cell:
#
#   'auto'    Resume from the latest checkpoint in CHECKPOINT_DIR if one
#             exists; otherwise start from scratch.  (default, safe for
#             Colab disconnect / reconnect)
#
#   'resume'  Same as 'auto' but raises an error when no checkpoint is found,
#             so you never silently start over.
#
#   'scratch' Delete any existing checkpoints in CHECKPOINT_DIR and train
#             from scratch.  Use this when you change MAX_SEQ_LEN, prompt
#             schema, or any hyperparameter that makes old checkpoints
#             incompatible with the current run.
# ---------------------------------------------------------------------------
TRAIN_MODE = 'auto'   # <-- change to 'scratch' or 'resume' as needed
# New seeds (123, 2024, 7, 99): use TRAIN_MODE='scratch' so old checkpoints cannot mix.


def _latest_checkpoint(output_dir: str):
    root = Path(output_dir)
    if not root.is_dir():
        return None
    ckpts = sorted(
        (p for p in root.iterdir() if p.is_dir() and p.name.startswith('checkpoint-')),
        key=lambda p: int(p.name.rsplit('-', 1)[-1]),
    )
    return str(ckpts[-1]) if ckpts else None


if SKIP_TRAINING:
    print('SKIP_TRAINING: not running SFTTrainer.')
elif TRAIN_MODE == 'scratch':
    if Path(CHECKPOINT_DIR).exists():
        shutil.rmtree(CHECKPOINT_DIR)
        print(f'Deleted existing checkpoints in {CHECKPOINT_DIR}')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    RESUME_CKPT = None
    print('Training from scratch.')
elif TRAIN_MODE == 'resume':
    RESUME_CKPT = _latest_checkpoint(CHECKPOINT_DIR)
    if not RESUME_CKPT:
        raise RuntimeError(
            f"TRAIN_MODE='resume' but no checkpoint found in {CHECKPOINT_DIR}. "
            "Change TRAIN_MODE to 'scratch' to start a new run."
        )
    print('Resuming from checkpoint:', RESUME_CKPT)
else:  # 'auto'
    RESUME_CKPT = _latest_checkpoint(CHECKPOINT_DIR)
    if RESUME_CKPT:
        print('Auto-resuming from checkpoint:', RESUME_CKPT)
    else:
        print('No checkpoint found; training from scratch.')

if SKIP_TRAINING:
    pass
else:
    _sft_config_params = inspect.signature(SFTConfig.__init__).parameters
    _NUM_EPOCHS = 2
    _TRAIN_BS = 16
    _GRAD_ACCUM = 2
    _steps_per_epoch = max(1, len(ds['train']) // (_TRAIN_BS * _GRAD_ACCUM))
    _total_steps = _steps_per_epoch * _NUM_EPOCHS
    _warmup_steps = max(1, int(_total_steps * 0.1))
    print(f'estimated train steps: {_total_steps} (warmup_steps={_warmup_steps})')

    _sft_kwargs = dict(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=_NUM_EPOCHS,
        per_device_train_batch_size=_TRAIN_BS,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=_GRAD_ACCUM,
        learning_rate=1e-4,
        lr_scheduler_type='cosine',
        weight_decay=0.0,
        bf16=True,
        logging_steps=20,
        eval_strategy='epoch',
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        packing=False,
        completion_only_loss=True,
        report_to='none',
        seed=RUN_SEED,
    )
    if 'warmup_steps' in _sft_config_params:
        _sft_kwargs['warmup_steps'] = _warmup_steps
    elif 'warmup_ratio' in _sft_config_params:
        _sft_kwargs['warmup_ratio'] = 0.1
    if 'max_length' in _sft_config_params:
        _sft_kwargs['max_length'] = MAX_SEQ_LEN
    elif 'max_seq_length' in _sft_config_params:
        _sft_kwargs['max_seq_length'] = MAX_SEQ_LEN
    sft_args = SFTConfig(**_sft_kwargs)

    _sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    _trainer_kwargs = dict(
        model=model,
        args=sft_args,
        train_dataset=ds['train'],
        eval_dataset=ds['validation'],
    )
    if 'processing_class' in _sft_trainer_params:
        _trainer_kwargs['processing_class'] = tokenizer
    elif 'tokenizer' in _sft_trainer_params:
        _trainer_kwargs['tokenizer'] = tokenizer

    trainer = SFTTrainer(**_trainer_kwargs)
    trainer.train(resume_from_checkpoint=RESUME_CKPT)

In [ ]:
from pathlib import Path
import shutil

SEED_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
if SKIP_TRAINING:
    print('SKIP_TRAINING: reusing adapter at', FINAL_ADAPTER_DIR)
else:
    os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
    trainer.save_model(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    print('LoRA adapter saved to', FINAL_ADAPTER_DIR)

src = Path(FINAL_ADAPTER_DIR)
if src.resolve() != SEED_ADAPTER_DIR.resolve():
    shutil.copytree(src, SEED_ADAPTER_DIR, dirs_exist_ok=True)
    print('Copied adapter to', SEED_ADAPTER_DIR)

_meta = {
    'base_model': MODEL_NAME,
    'final_adapter_dir': FINAL_ADAPTER_DIR,
    'seed_adapter_dir': str(SEED_ADAPTER_DIR),
    'checkpoint_dir': CHECKPOINT_DIR,
    'seed': RUN_SEED,
    'system_prompt': SYSTEM_PROMPT,
    'max_seq_len': MAX_SEQ_LEN,
    'prompt_schema': PROMPT_SCHEMA_VERSION,
    'prompt_prob_decimals': PROMPT_PROB_DECIMALS,
    'train_batch_size': 16,
    'gradient_accumulation_steps': 2,
}
for _meta_path in (ARTIFACTS_DIR / 'lora_train_meta.json', SEED_ADAPTER_DIR / 'lora_train_meta.json'):
    with open(_meta_path, 'w', encoding='utf-8') as f:
        json.dump(_meta, f, ensure_ascii=False, indent=2)
print('Wrote', ARTIFACTS_DIR / 'lora_train_meta.json', 'and', SEED_ADAPTER_DIR / 'lora_train_meta.json')
print('Trainer checkpoints (for resume) remain in', CHECKPOINT_DIR)

In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()